In [1]:
import sys
sys.path.append("../src")

In [2]:
from Fasttext.Connect_Database import connect
from preprocessing import clean,stopword
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split
import csv

In [3]:
app = connect()
df = pd.DataFrame(app.load_data_VnExpress())

c:\Users\admin\Data\anaconda\Lib\site-packages\pymongo\pyopenssl_context.py:355: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  _crypto.X509.from_cryptography(x509.load_der_x509_certificate(cert))


In [4]:
df[['topic', 'subtopic', 'content']].head(10)

,topic,subtopic,content
0,Thời sự,Chính trị,"Chiều 19/8, Tổng Bí thư Tô Lâm làm việc với Ba..."
1,Thời sự,Chính trị,"Tổng Bí thư nhấn mạnh việc chuẩn bị, tổ chức c..."
2,Thời sự,Chính trị,Người đứng đầu Đảng đề nghị các cơ quan triển ...
3,Thời sự,Chính trị,"Đồng thời, Tổng Bí thư lưu ý công tác đón tiếp..."
4,Thời sự,Chính trị,"Tổng Bí thư lưu ý trong dịp này, đặc biệt là v..."
5,Thời sự,Chính trị,Theo báo cáo của Ban Tuyên giáo và Dân vận Tru...
6,Thời sự,Chính trị,Các hạng mục hạ tầng trọng yếu chuẩn bị cho sự...
7,Thời sự,Kỷ nguyên mới,"Tháng 8/1945, cuộc tổng khởi nghĩa giành chính..."
8,Thời sự,Chính trị,"Để chuẩn bị cho lễ diễu binh, diễu hành kỷ niệ..."
9,Thời sự,Chính trị,"Riêng lực lượng diễu binh, diễu hành gồm 4 khố..."


In [5]:
def process_texts(texts):
    cleaned = [clean(str(t)) for t in texts]
    processed = stopword(cleaned)
    return processed

In [6]:
df['clean_content'] = process_texts(df['content'])

In [7]:
df = df[
    ~df['subtopic'].isin(['Ảnh']) &
    ~df['topic'].isin(['Thể thao'])
]
counts = df['subtopic'].value_counts()
df = df[df['subtopic'].isin(counts[counts >= 1000].index)]

In [8]:
df['label'] = '__label__' + df['topic'].str.replace(' ', '_') + '__' + df['subtopic'].str.replace(' ', '_')

In [9]:
df.head(10)

,content,subtopic,title,topic,clean_content,label
0,"Chiều 19/8, Tổng Bí thư Tô Lâm làm việc với Ba...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,chiều 19 8 tổng bí thư tô lâm ban đạo trung ươ...,__label__Thời_sự__Chính_trị
1,"Tổng Bí thư nhấn mạnh việc chuẩn bị, tổ chức c...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,tổng bí thư nhấn chuẩn tổ chức hoạt động kỷ ni...,__label__Thời_sự__Chính_trị
2,Người đứng đầu Đảng đề nghị các cơ quan triển ...,Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,đứng đầu đảng đề nghị quan triển khai hiệu kế ...,__label__Thời_sự__Chính_trị
3,"Đồng thời, Tổng Bí thư lưu ý công tác đón tiếp...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,đồng thời tổng bí thư lưu công tác đón tiếp đạ...,__label__Thời_sự__Chính_trị
4,"Tổng Bí thư lưu ý trong dịp này, đặc biệt là v...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,tổng bí thư lưu dịp đặc biệt du đổ thủ đô hà n...,__label__Thời_sự__Chính_trị
5,Theo báo cáo của Ban Tuyên giáo và Dân vận Tru...,Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,báo cáo ban tuyên giáo dân vận trung ương công...,__label__Thời_sự__Chính_trị
6,Các hạng mục hạ tầng trọng yếu chuẩn bị cho sự...,Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,hạng mục hạ tầng trọng yếu chuẩn kiện hoàn thà...,__label__Thời_sự__Chính_trị
7,"Tháng 8/1945, cuộc tổng khởi nghĩa giành chính...",Kỷ nguyên mới,Mỗi người dân được tặng 100.000 đồng dịp Quốc ...,Thời sự,8 1945 tổng khởi nghĩa giành quyền thành công ...,__label__Thời_sự__Kỷ_nguyên_mới
8,"Để chuẩn bị cho lễ diễu binh, diễu hành kỷ niệ...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,chuẩn lễ diễu binh diễu hành kỷ niệm 80 mạng t...,__label__Thời_sự__Chính_trị
9,"Riêng lực lượng diễu binh, diễu hành gồm 4 khố...",Chính trị,Tổng Bí thư: Lễ diễu binh phải thể hiện được v...,Thời sự,lực diễu binh diễu hành 4 khối nghi trượng 43 ...,__label__Thời_sự__Chính_trị


In [10]:
df['label'].value_counts()

label
__label__Sức_khỏe__Tin_tức                       7460
__label__Thế_giới__Tư_liệu                       4173
__label__Thế_giới__Quân_sự                       4067
__label__Kinh_doanh__Doanh_nghiệp                3848
__label__Sức_khỏe__Sống_khỏe                     3564
__label__Giải_trí__Giới_sao                      3494
__label__Đời_sống__Nhịp_sống                     3303
__label__Bất_động_sản__Thị_trường                2781
__label__Kinh_doanh__Quốc_tế                     2728
__label__Thế_giới__Cuộc_sống_đó_đây              2715
__label__Giáo_dục__Tin_tức                       2620
__label__Thế_giới__Phân_tích                     2519
__label__Pháp_luật__Hồ_sơ_phá_án              2239
__label__Giải_trí__Phim                          2184
__label__Sức_khỏe__Các_bệnh                      2131
__label__Thời_sự__80_năm_Quốc_khánh              2089
__label__Kinh_doanh__Vĩ_mô                       1987
__label__Đời_sống__Bài_học_sống                  1965
__label__Du_lịch__Điểm

In [11]:
df['format'] = df['label'] + ' ' + df['clean_content']
max_count= 1500
balanced_data = (df.groupby('label', group_keys=False, as_index=False).apply(lambda x: x.sample(max_count, replace=True)).reset_index(drop=True))

train, test = train_test_split(balanced_data, test_size=0.2, random_state=42)
train['format'].to_csv("../Data/processed/dataset_train.txt",index=False, header=False,sep='\n', quoting=csv.QUOTE_NONE, escapechar='\\')
test['format'].to_csv("../Data/processed/dataset_test.txt",index=False, header=False,sep='\n', quoting=csv.QUOTE_NONE, escapechar='\\')
df['format'].to_csv('../Data/processed/train.txt', index=False, header=False, sep='\n',quoting=csv.QUOTE_NONE,escapechar='\\')

C:\Users\admin\AppData\Local\Temp\ipykernel_35136\3722749065.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_data = (df.groupby('label', group_keys=False, as_index=False).apply(lambda x: x.sample(max_count, replace=True)).reset_index(drop=True))


In [12]:
pd.set_option('display.max_rows', None)
print(balanced_data['label'].value_counts())

label
__label__Bất_động_sản__Không_gian_sống           1500
__label__Bất_động_sản__Thị_trường                1500
__label__Kinh_doanh__Vĩ_mô                       1500
__label__Pháp_luật__Hồ_sơ_phá_án              1500
__label__Pháp_luật__Tư_vấn                       1500
__label__Sức_khỏe__Các_bệnh                      1500
__label__Sức_khỏe__Sống_khỏe                     1500
__label__Sức_khỏe__Tin_tức                       1500
__label__Sức_khỏe__Vaccine                       1500
__label__Thế_giới__Cuộc_sống_đó_đây              1500
__label__Thế_giới__Phân_tích                     1500
__label__Thế_giới__Quân_sự                       1500
__label__Thế_giới__Tư_liệu                       1500
__label__Thời_sự__80_năm_Quốc_khánh              1500
__label__Thời_sự__Chính_trị                      1500
__label__Thời_sự__Giao_thông                     1500
__label__Thời_sự__Kỷ_nguyên_mới                  1500
__label__Xe__Thị_trường                          1500
__label__Xe__V-Car    

In [13]:
model = fasttext.train_supervised(
    input="../Data/processed/dataset_train.txt",
    lr=0.1,
    epoch=100,
    wordNgrams=3,
    dim=100,
    loss='softmax'
)

In [14]:
result = model.test("../Data/processed/dataset_test.txt")
print(f"Số mẫu test: {result[0]}")
print(f"Độ chính xác (precision): {result[1]:.4f}")
print(f"Độ bao phủ (recall): {result[2]:.4f}")
print(f"F1-score: {2 * result[1] * result[2] / (result[1] + result[2]):.4f}")

Số mẫu test: 13200
Độ chính xác (precision): 0.7970
Độ bao phủ (recall): 0.7970
F1-score: 0.7970


In [15]:
model.save_model("../models/Fasttext/model_1.bin")

In [16]:
df_Reddit = pd.DataFrame(app.load_data())
df_Reddit['clean_content'] = process_texts(df_Reddit['content'])

In [17]:
df_Reddit.head(5)

,link,author,comments,content,date,image_link,subreddit,title,upvotes,video_link,clean_content
0,https://www.reddit.com/r/TroChuyenLinhTinh/com...,Friendly-Lie5849,37,Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoa...,2025-08-22 07:25:50,None,TroChuyenLinhTinh,Những gì đang diễn ra ở Trung Quốc đang dần lặ...,157,None,phim tuyên truyền chủ nghĩa dân tộc cực đoan m...
1,https://www.reddit.com/r/vozforums/comments/1m...,TinNT0409,1,"Chào mọi người, đợt này mình có dùng Mathpix đ...",2025-08-22 07:23:33,https://external-preview.redd.it/kqhxR96oyl5gx...,vozforums,Phần mềm quét công thức sang Word miễn phí tha...,2,None,chào đợt mathpix quét công thức đc 10 lượt viế...
2,https://www.reddit.com/r/TroChuyenLinhTinh/com...,Weekly_Top7078,3,Trước kì kinh bao nhiêu ngày thì an toàn?,2025-08-22 07:07:46,None,TroChuyenLinhTinh,Ngày an toàn,0,None,kì kinh bao nhiêu an toàn
3,https://www.reddit.com/r/VietNamNation/comment...,Ambitious-Fan-9831,18,ờm bỏ qua tỉ lệ gái đẹp ra thì tao thấy bề nổi...,2025-08-22 07:04:09,None,VietNamNation,Thái Lan có phải là 1 hình mẫu nước đáng sống ...,16,None,ờm tỉ lệ gái đẹp tao bề nổi thái lọ cánh tả số...
4,https://www.reddit.com/r/vozforums/comments/1m...,Nicklas0704,3,"Hi các bác, e làm mmo có dư 1 ít, cụ thể là kh...",2025-08-22 14:03:21,None,vozforums,Hiện tại nên bỏ tiền vào đâu ?,0,None,hi e mmo dư 1 cụ thể 10 tỉ tham gia đầu hiện e...


In [18]:
df_Reddit = df_Reddit[
    df_Reddit['clean_content'].notna() &                   
    (df_Reddit['clean_content'].str.strip().str.lower() != 'none') &
    (df_Reddit['clean_content'].str.strip() != '')
]

In [19]:
df_Reddit = df_Reddit[['author', 'title','clean_content', 'content']]
df_Reddit.head(2)

,author,title,clean_content,content
0,Friendly-Lie5849,Những gì đang diễn ra ở Trung Quốc đang dần lặ...,phim tuyên truyền chủ nghĩa dân tộc cực đoan m...,Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoa...
1,TinNT0409,Phần mềm quét công thức sang Word miễn phí tha...,chào đợt mathpix quét công thức đc 10 lượt viế...,"Chào mọi người, đợt này mình có dùng Mathpix đ..."


In [20]:
labels = []
scores = []
for text in df_Reddit['clean_content']:
    if not isinstance(text, str):
        text = ""
    text_clean = text.replace("\n", " ").strip()
    prediction = model.predict(text_clean)
    labell = prediction[0][0].replace("__label__", "")
    score = prediction[1][0]
    labels.append(labell)
    scores.append(score)
df_Reddit['label'] = labels
df_Reddit['score'] = scores


In [21]:
df_Reddit.head(10)

,author,title,clean_content,content,label,score
0,Friendly-Lie5849,Những gì đang diễn ra ở Trung Quốc đang dần lặ...,phim tuyên truyền chủ nghĩa dân tộc cực đoan m...,Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoa...,Giải_trí__Phim,0.477290
1,TinNT0409,Phần mềm quét công thức sang Word miễn phí tha...,chào đợt mathpix quét công thức đc 10 lượt viế...,"Chào mọi người, đợt này mình có dùng Mathpix đ...",Giáo_dục__Tin_tức,0.458249
2,Weekly_Top7078,Ngày an toàn,kì kinh bao nhiêu an toàn,Trước kì kinh bao nhiêu ngày thì an toàn?,Thế_giới__Cuộc_sống_đó_đây,0.363906
3,Ambitious-Fan-9831,Thái Lan có phải là 1 hình mẫu nước đáng sống ...,ờm tỉ lệ gái đẹp tao bề nổi thái lọ cánh tả số...,ờm bỏ qua tỉ lệ gái đẹp ra thì tao thấy bề nổi...,Thế_giới__Tư_liệu,0.357446
4,Nicklas0704,Hiện tại nên bỏ tiền vào đâu ?,hi e mmo dư 1 cụ thể 10 tỉ tham gia đầu hiện e...,"Hi các bác, e làm mmo có dư 1 ít, cụ thể là kh...",Kinh_doanh__Hàng_hóa,0.553686
5,Chunghiacanhanvidai,Đừng Đánh Giá Tự Do Xã Hội Bằng Giáo Điều Phươ...,đừng đánh giá xã hội giáo phương tây trần trụi...,Đừng Đánh Giá Tự Do Xã Hội Bằng Giáo Điều Phươ...,Giải_trí__Sách,0.538289
7,117431853211,Xin mọi người cho em lời khuyên,nay 17 sống bình chí thể coi êm đềm bất kỳ biế...,"Năm nay em 17 tuổi, cuộc sống rất bình thường,...",Đời_sống__Tổ_ấm,0.314655
8,LeeTuneVane,Con BHP rồi mẹ ạ .Nếu may mắn thì gia đình mìn...,bhp mẹ may mắn gia đình kinh tế mẹ bát cơm can...,Con BHP rồi mẹ ạ .Nếu may mắn thì gia đình mìn...,Đời_sống__Tổ_ấm,0.672545
10,dahoodcashseller,Sỹ con chạy xe quá lẹ bắt chước cha,welp ae kết tụi chạy 70km h ko đội nón tông xo...,Welp chắc ae cũng biết cái kết rồi tụi này chạ...,Thế_giới__Cuộc_sống_đó_đây,0.360535
11,fishmeaterm,Kêu gọi toàn dân VietNamNation vào trị bọn Pod...,kêu gọi toàn dân đảo vietnamnation trị bọn pod...,Kêu gọi toàn dân đảo VietNamNation vào trị bọn...,Kinh_doanh__NetZero,0.507341


In [22]:
for idx, row in df_Reddit.head(50).iterrows():
    print(f"Author: {row['author']}")
    print(f"Title: {row['title']}")
    print(f"Content: {row['content'][:200]}...")
    print(f"Label: {row['label']}")
    print(f"Score: {row['score']:.3f}")
    print("-"*60)

Author: Friendly-Lie5849
Title: Những gì đang diễn ra ở Trung Quốc đang dần lặp lại ở Việt Nam
Content: Bộ phim tuyên truyền chủ nghĩa dân tộc cực đoan Mưa Đỏ aka Máu Kinh đang được truyền thông, dư luận viên, bò đỏ, Sô-Vanh con tung hê hết lời. Phim hiện đã thu về 28 tỷ đồng vào ngày đầu công chiếu là ...
Label: Giải_trí__Phim
Score: 0.477
------------------------------------------------------------
Author: TinNT0409
Title: Phần mềm quét công thức sang Word miễn phí thay thế Mathpix, Mathtype, Equation mặc định trong Word
Content: Chào mọi người, đợt này mình có dùng Mathpix để quét công thức nhưng đc có 10 lượt nên Viết luôn app thay thế miễn phí [AuraLateX](https://auravsoftware.com/chuyen-cong-thuc-toan-sang-word/)...
Label: Giáo_dục__Tin_tức
Score: 0.458
------------------------------------------------------------
Author: Weekly_Top7078
Title: Ngày an toàn
Content: Trước kì kinh bao nhiêu ngày thì an toàn?...
Label: Thế_giới__Cuộc_sống_đó_đây
Score: 0.364
-------------------------

In [23]:
df_Reddit['label'].value_counts()

label
Giải_trí__Sách                          1464
Thế_giới__Cuộc_sống_đó_đây              1431
Đời_sống__Bài_học_sống                  1100
Đời_sống__Nhịp_sống                      810
Đời_sống__Tổ_ấm                          781
Thời_sự__80_năm_Quốc_khánh               650
Thế_giới__Tư_liệu                        616
Pháp_luật__Hồ_sơ_phá_án               614
Giáo_dục__Du_học                         534
Khoa_học_công_nghệ__AI                   485
Giáo_dục__Tin_tức                        464
Sức_khỏe__Tin_tức                        359
Thời_sự__Chính_trị                       313
Sức_khỏe__Sống_khỏe                      261
Giải_trí__Giới_sao                       256
Khoa_học_công_nghệ__Thiết_bị             254
Xe__V-Car                                245
Giải_trí__Phim                           224
Thời_sự__Kỷ_nguyên_mới                   214
Khoa_học_công_nghệ__Chuyển_đổi_số        207
Thế_giới__Phân_tích                      203
Kinh_doanh__Kinh_tế_vùng                 192
Du_l